# K-FRAG clean watermark baseline — UNTRAINED SMOKE TEST
This notebook performs one forward pass only. It does not train the model.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
def find_repository_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'kfrag').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the K-FRAG repository root')
repository_root = find_repository_root(Path.cwd())
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))
from kfrag.crypto import ProvenanceToken, create_packets
from kfrag.data import CocoImageDataset
from kfrag.payload.regional_tensor import batch_packets_to_grid
from kfrag.models import CleanWatermarkSystem
from kfrag.models.losses import clean_watermark_loss, bit_accuracy, psnr

In [ ]:
print('UNTRAINED SMOKE TEST')
image_directory = repository_root / 'data/raw/coco_val2017_100'
dataset = CocoImageDataset(image_directory)
images = next(iter(DataLoader(dataset, batch_size=4, shuffle=False, num_workers=0)))['image']
secret_key = b'notebook-smoke-key'  # Used locally; deliberately never printed.
tokens = [ProvenanceToken.generate(issuer_id=1) for _ in range(4)]
payload = batch_packets_to_grid([create_packets(token, secret_key) for token in tokens])
model = CleanWatermarkSystem(base_channels=32, message_channels=32, residual_alpha=0.02)
model.eval()
with torch.no_grad():
    output = model(images, payload)
    losses = clean_watermark_loss(output['payload_logits'], payload, images, output['watermarked_image'], output['residual'])
    smoke_psnr = psnr(images, output['watermarked_image'])
    smoke_bits = bit_accuracy(output['payload_logits'], payload)
print('UNTRAINED SMOKE TEST shapes:', {k: tuple(v.shape) for k, v in output.items()})
print(f'PSNR: {smoke_psnr.item():.2f} dB')
print(f'Untrained bit accuracy: {smoke_bits.item():.4f}')
print(f'Total loss: {losses["total_loss"].item():.4f}')

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for index in range(4):
    axes[0, index].imshow(images[index].permute(1, 2, 0).clamp(0, 1))
    axes[0, index].set_title('Original')
    axes[1, index].imshow(output['watermarked_image'][index].permute(1, 2, 0).clamp(0, 1))
    axes[1, index].set_title('Watermarked')
    visual_residual = (output['residual'][index] * 10 + 0.5).permute(1, 2, 0).clamp(0, 1)
    axes[2, index].imshow(visual_residual)
    axes[2, index].set_title('Residual ×10 (visual only)')
for axis in axes.flat:
    axis.axis('off')
plt.suptitle('UNTRAINED SMOKE TEST')
plt.tight_layout()